# NLP Workflow - LDA Topic Modelling on full dataset using optimum configuration

## This Notebook Covers
- Retry Bag-of-words for preliminary comparison
- Use LDA with bigrams and optimum k=50 for topic modelling
- Review topic distribution and bigrams in topics

## Libraries Used
- **Python:** pandas, tqdm, sklearn

---
## 1. Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import nltk
import re
import string
import pickle
import json
import time
from tqdm import tqdm

# NLP libraries
from nltk.corpus import stopwords
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split

# Visualization settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

---
## 2. Load Preprocessed Data

In [2]:
print("=" * 70)
print("LOADING PREPROCESSED COMPLAINT DATA")
print("=" * 70)

# Load data
df = pd.read_csv("data/complaints_with_nlp_features_lda.csv", 
                 engine="python",
                 on_bad_lines="skip")

# Remove unnamed index if exists
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print(f"\nLoaded {len(df):,} complaints")
print(f"   Columns: {len(df.columns)}")
print(f"   Date range: {df['Date received'].min()} to {df['Date received'].max()}")

# Show sample
df.head(3)

LOADING PREPROCESSED COMPLAINT DATA

Loaded 1,399,222 complaints
   Columns: 31
   Date range: 01/01/23 to 12/31/25


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,narrative_clean,narrative_no_stopwords,sentiment_compound,sentiment_neg,sentiment_neu,sentiment_pos,urgency_score,churn_intent,loyalty_score,text_length,word_count,narrative_cleaned,narrative_for_lda
0,01/20/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,I am writing to have the following information...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,92345,NaN,Consent provided,Web,01/20/25,Closed with non-monetary relief,Yes,NaN,11588109,i am writing to have the following information...,writing following information removed credit f...,0.9430,0.037,0.794,0.169,0.0,0.4,0.0,662,119,writing following information removed credit f...,writing have the following information removed...
1,07/03/24,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,I am a victim of identity theft. Please delete...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,32824,NaN,Consent provided,Web,07/03/24,Closed with non-monetary relief,Yes,NaN,9416677,i am a victim of identity theft. please delete...,victim identity theft please delete remove ite...,0.2732,0.100,0.760,0.139,0.0,0.0,0.0,370,68,victim identity theft please delete remove ite...,victim identity theft. please delete remove th...
2,09/14/25,Vehicle loan or lease,Loan,Incorrect information on your report,Information belongs to someone else,"My name is XXXX XXXX, and I am formally disput...",NaN,"SANTANDER HOLDINGS USA, INC.",PA,19143,NaN,Consent provided,Web,09/14/25,Closed with explanation,Yes,NaN,15930829,"my name is xxxx xxxx, and i am formally disput...",name xxxx xxxx formally disputing fraudulent a...,-0.9042,0.151,0.766,0.083,1.0,0.4,0.0,830,140,name formally disputing fraudulent auto loan s...,"name , and formally disputing fraudulent auto ..."


---
## 3. Bag-of-Words (TF-IDF) Feature Extraction

Extract top 250 TF-IDF features after CFPB redaction cleaning.

In [4]:
print("=" * 70)
print("BAG-OF-WORDS (TF-IDF) FEATURE EXTRACTION")
print("=" * 70)

# Check cleaning effectiveness
avg_len_before = df['narrative_no_stopwords'].str.len().mean()
avg_len_after = df['narrative_clean'].str.len().mean()
reduction_pct = (avg_len_before - avg_len_after) / avg_len_before * 100

print(f"   Avg text length before: {avg_len_before:.0f} chars")
print(f"   Avg text length after:  {avg_len_after:.0f} chars")
print(f"   Reduction: {reduction_pct:.1f}%")

# Create TF-IDF vectors
print("\nCreating TF-IDF vectors (bigrams for BoW representation)...")

tfidf_vectorizer = TfidfVectorizer(
    max_features=250,
    ngram_range=(1, 2),  # Unigrams + bigrams for BoW
    min_df=20,
    max_df=0.8,
    sublinear_tf=True,
    use_idf=True
)

X_tfidf = tfidf_vectorizer.fit_transform(df['narrative_clean'])
feature_names = tfidf_vectorizer.get_feature_names_out()

print(f"   Created TF-IDF matrix: {X_tfidf.shape}")
print(f"      Documents: {X_tfidf.shape[0]:,}")
print(f"      Features: {X_tfidf.shape[1]}")

# Top features
avg_tfidf = np.asarray(X_tfidf.mean(axis=0)).flatten()
top_indices = avg_tfidf.argsort()[-20:][::-1]

print(f"\nTop 20 TF-IDF features:")
for i, idx in enumerate(top_indices, 1):
    print(f"   {i:2d}. {feature_names[idx]:30s}: {avg_tfidf[idx]:.4f}")

# Save vectorizer
with open('models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print("\nSaved: models/tfidf_vectorizer.pkl")

BAG-OF-WORDS (TF-IDF) FEATURE EXTRACTION
   Avg text length before: 732 chars
   Avg text length after:  1008 chars
   Reduction: -37.6%

Creating TF-IDF vectors (bigrams for BoW representation)...
   Created TF-IDF matrix: (1399222, 250)
      Documents: 1,399,222
      Features: 250

Top 20 TF-IDF features:
    1. xxxx                          : 0.1205
    2. xxxx xxxx                     : 0.0975
    3. of                            : 0.0826
    4. credit                        : 0.0765
    5. this                          : 0.0683
    6. that                          : 0.0676
    7. report                        : 0.0641
    8. on                            : 0.0621
    9. is                            : 0.0617
   10. have                          : 0.0613
   11. not                           : 0.0605
   12. in                            : 0.0591
   13. my credit                     : 0.0589
   14. information                   : 0.0554
   15. xx                            : 0.0551

---
## 4. Topic Modeling with LDA

**Objective:** Extract 50 interpretable topics using LDA  
**Method:** Use bigram with k=50

### 4.1 Perplexity Calculation - Bigram
 
**Method:** Train both models, compare test perplexity (lower = better)

In [7]:
print("=" * 70)
print("TRAIN/TEST SPLIT FOR PERPLEXITY")
print("=" * 70)

print(f"\nFull dataset: {len(df):,} complaints")
print(f"Split: 80% train, 20% test")
print(f"Random seed: 42")

train_texts, test_texts = train_test_split(
    df['narrative_for_lda'],
    test_size=0.2,
    random_state=42
)

print(f"\nSplit complete:")
print(f"   Train: {len(train_texts):,} complaints")
print(f"   Test:  {len(test_texts):,} complaints")

TRAIN/TEST SPLIT FOR PERPLEXITY

Full dataset: 1,399,222 complaints
Split: 80% train, 20% test
Random seed: 42

Split complete:
   Train: 1,119,377 complaints
   Test:  279,845 complaints


In [8]:
# ─── BIGRAM MODEL ───
print("\n" + "─" * 70)
print("BIGRAM LDA")
print("─" * 70)

vectorizer_bigram = CountVectorizer(
    max_features=8000,
    min_df=20,
    max_df=0.9,
    ngram_range=(1, 2),  # Unigrams + bigrams
    token_pattern=r'(?u)\b\w+\b',
    stop_words='english'
)

print("\nBuilding vocabulary...")
train_bow_bi = vectorizer_bigram.fit_transform(train_texts)
test_bow_bi = vectorizer_bigram.transform(test_texts)

vocab_bi = vectorizer_bigram.get_feature_names_out()
bigrams = [f for f in vocab_bi if ' ' in f]
print(f"   Vocabulary: {len(vocab_bi):,} terms ({len(bigrams):,} bigrams)")

print("\nTraining LDA...")
lda_bigram = LatentDirichletAllocation(
    n_components=50,
    max_iter=20,
    learning_method='online',
    learning_offset=50.0,
    batch_size=2048,
    evaluate_every=-1,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

start = time.time()
lda_bigram.fit(train_bow_bi)
train_time_bi = time.time() - start

# Calculate perplexity
train_perp_bi = lda_bigram.perplexity(train_bow_bi)
test_perp_bi = lda_bigram.perplexity(test_bow_bi)

print(f"\n   Training time: {train_time_bi:.1f}s")
print(f"   Train perplexity: {train_perp_bi:.2f}")
print(f"   Test perplexity:  {test_perp_bi:.2f}")


──────────────────────────────────────────────────────────────────────
BIGRAM LDA
──────────────────────────────────────────────────────────────────────

Building vocabulary...
   Vocabulary: 8,000 terms (5,045 bigrams)

Training LDA...
iteration: 1 of max_iter: 20
iteration: 2 of max_iter: 20
iteration: 3 of max_iter: 20
iteration: 4 of max_iter: 20
iteration: 5 of max_iter: 20
iteration: 6 of max_iter: 20
iteration: 7 of max_iter: 20
iteration: 8 of max_iter: 20
iteration: 9 of max_iter: 20
iteration: 10 of max_iter: 20
iteration: 11 of max_iter: 20
iteration: 12 of max_iter: 20
iteration: 13 of max_iter: 20
iteration: 14 of max_iter: 20
iteration: 15 of max_iter: 20
iteration: 16 of max_iter: 20
iteration: 17 of max_iter: 20
iteration: 18 of max_iter: 20
iteration: 19 of max_iter: 20
iteration: 20 of max_iter: 20

   Training time: 2697.4s
   Train perplexity: 597.34
   Test perplexity:  601.68


In [9]:
vocab_bi

array(['1028a', '1099c', '15usc', ..., 'zelle numerous', 'zelles', 'zero'],
      shape=(8000,), dtype=object)

In [10]:
df_bi_vocab = pd.DataFrame(vocab_bi, columns=["values"])
df_bi_vocab.to_csv("data/vocab_bi_8k.csv", index=False)

### 4.2 Extract Topics

In [12]:
print("=" * 70)
print("FINAL TOPIC EXTRACTION (FULL DATASET)")
print("=" * 70)

final_vectorizer = CountVectorizer(
    max_features=8000,
    min_df=20,
    max_df=0.9,
    ngram_range=(1, 2),  # Unigrams + bigrams
    token_pattern=r'(?u)\b\w+\b',
    stop_words='english'
)

print(f"\nProcessing {len(df):,} complaints...")
X_counts = final_vectorizer.fit_transform(df['narrative_for_lda'])
print(f"   Vocabulary size: {X_counts.shape[1]:,} terms")

# Train final LDA
print("\nTraining final LDA model")
final_lda = LatentDirichletAllocation(
    n_components=50,
    max_iter=20,
    learning_method='online',
    learning_offset=50.0,
    batch_size=2048,
    evaluate_every=-1,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

final_lda.fit(X_counts)
print("\n   LDA training complete")

FINAL TOPIC EXTRACTION (FULL DATASET)

Processing 1,399,222 complaints...
   Vocabulary size: 8,000 terms

Training final LDA model
iteration: 1 of max_iter: 20
iteration: 2 of max_iter: 20
iteration: 3 of max_iter: 20
iteration: 4 of max_iter: 20
iteration: 5 of max_iter: 20
iteration: 6 of max_iter: 20
iteration: 7 of max_iter: 20
iteration: 8 of max_iter: 20
iteration: 9 of max_iter: 20
iteration: 10 of max_iter: 20
iteration: 11 of max_iter: 20
iteration: 12 of max_iter: 20
iteration: 13 of max_iter: 20
iteration: 14 of max_iter: 20
iteration: 15 of max_iter: 20
iteration: 16 of max_iter: 20
iteration: 17 of max_iter: 20
iteration: 18 of max_iter: 20
iteration: 19 of max_iter: 20
iteration: 20 of max_iter: 20

   LDA training complete


In [14]:
# Display topics
print("\n" + "=" * 70)
print("DISCOVERED TOPICS")
print("=" * 70)

feature_names = final_vectorizer.get_feature_names_out()
n_top_words = 20

topics = []
for topic_idx, topic in enumerate(final_lda.components_):
    top_indices = topic.argsort()[-n_top_words:][::-1]
    top_words = [feature_names[i] for i in top_indices]
    topics.append(top_words)
    
    print(f"\nTopic {topic_idx + 1}:")
    print(f"  {', '.join(top_words)}")

# Assign topics to documents
doc_topics = final_lda.transform(X_counts)
dominant_topics = doc_topics.argmax(axis=1)

df['dominant_topic'] = dominant_topics

print("\n" + "─" * 70)
print("Topic Distribution:")
print("─" * 70)

topic_counts = pd.Series(dominant_topics).value_counts().sort_index()
for topic_idx, count in topic_counts.items():
    pct = count / len(dominant_topics) * 100
    print(f"  Topic {topic_idx + 1}: {count:8,} ({pct:5.1f}%)")

# Save model
with open('models/lda_final_model.pkl', 'wb') as f:
    pickle.dump(final_lda, f)

with open('models/lda_final_vectorizer.pkl', 'wb') as f:
    pickle.dump(final_vectorizer, f)

print("\nSaved LDA model and vectorizer")


DISCOVERED TOPICS

Topic 1:
  information, consumer, report, inaccurate, credit, item, reasonable, disputed, reporting, reinvestigation, according, item information, accuracy, procedures, experian, promptly, reasonable procedures, unverifiable, consumer reporting, incomplete

Topic 2:
  late, payment, payments, late payment, late payments, account, credit, paid, reported, time, reporting, report, history, error, payment history, days, billing, reflect, credit report, inaccurate

Topic 3:
  usc, violation, credit, report, reporting, accounts, credit report, rights, inaccurate, according, information, accurate, violation usc, 1681i, delete, code, removed, usc 1681i, days, pursuant usc

Topic 4:
  account, bank, told, money, called, said, check, did, phone, funds, time, asked, received, days, just, sent, help, email, know, pay

Topic 5:
  balance, balance balance, owed, balance owed, items, original, original creditor, creditor, complaint, owed balance, bureaus, remove, inquired, credito

In [ ]:
print("=" * 70)
print("NLP FEATURE EXTRACTION (VADER)")
print("=" * 70)

# Merge LDA results back to main dataframe
df = df.merge(
    df_lda[['dominant_topic']],
    left_index=True,
    right_index=True,
    how='left'
)

# Merge cleaned text
df = df.merge(
    df[['narrative_clean']],
    left_index=True,
    right_index=True,
    how='left'
)

print(f"\nProcessing {len(df):,} complaints...")

In [ ]:
# Initialize VADER
vader = SentimentIntensityAnalyzer()

# Define keyword sets
urgency_words = {
    'urgent', 'immediately', 'asap', 'emergency', 'critical',
    'lawyer', 'attorney', 'sue', 'legal', 'court', 'report',
    'bbb', 'ftc', 'cfpb', 'complaint', 'authority'
}

churn_words = {
    'close', 'closing', 'cancel', 'canceling', 'terminate',
    'switch', 'switching', 'leave', 'leaving', 'done'
}

loyalty_words = {
    'year', 'years', 'decade', 'decades', 'loyal',
    'long time', 'longtime', 'customer'
}

print("\nKeyword sets defined:")
print(f"  Urgency:     {len(urgency_words)} keywords")
print(f"  Churn:       {len(churn_words)} keywords")
print(f"  Loyalty:     {len(loyalty_words)} keywords")

In [ ]:
def extract_features(text):
    """
    Extract NLP features from complaint text.
    
    Returns:
    --------
    dict with keys: sentiment_compound, sentiment_neg, sentiment_neu, 
                    sentiment_pos, urgency_score, churn_intent, 
                    loyalty_score, text_length, word_count
    """
    if not text or pd.isna(text):
        return {
            'sentiment_compound': 0.0,
            'sentiment_neg': 0.0,
            'sentiment_neu': 1.0,
            'sentiment_pos': 0.0,
            'urgency_score': 0.0,
            'churn_intent': 0.0,
            'loyalty_score': 0.0,
            'text_length': 0,
            'word_count': 0
        }
    
    # Sentiment
    scores = vader.polarity_scores(text)
    
    # Text stats
    text_lower = text.lower()
    words = text_lower.split()
    
    # Urgency (normalized by text length)
    urgency_count = sum(1 for word in urgency_words if word in text_lower)
    urgency_score = min(urgency_count / max(len(words) / 100, 1), 1.0)
    
    # Churn intent (tiered by phrase strength)
    if 'closing my account' in text_lower or 'close my account' in text_lower:
        churn_intent = 0.8
    elif 'cancel my account' in text_lower or 'terminate my account' in text_lower:
        churn_intent = 0.7
    elif any(word in text_lower for word in ['close', 'cancel', 'terminate']):
        churn_intent = 0.4
    elif any(word in text_lower for word in ['switch', 'leave', 'done']):
        churn_intent = 0.3
    else:
        churn_intent = 0.0
    
    # Loyalty
    loyalty_count = sum(1 for word in loyalty_words if word in text_lower)
    loyalty_score = min(loyalty_count / max(len(words) / 100, 1), 1.0)
    
    return {
        'sentiment_compound': scores['compound'],
        'sentiment_neg': scores['neg'],
        'sentiment_neu': scores['neu'],
        'sentiment_pos': scores['pos'],
        'urgency_score': urgency_score,
        'churn_intent': churn_intent,
        'loyalty_score': loyalty_score,
        'text_length': len(text),
        'word_count': len(words)
    }

print("\nExtracting features (this may take a few minutes)...")

In [ ]:
# Extract features with progress bar
features_list = []
batch_size = 100000

for i in range(0, len(df), batch_size):
    batch = df['narrative_clean'].iloc[i:i+batch_size]
    batch_features = batch.apply(extract_features)
    features_list.extend(batch_features.tolist())
    
    if (i + batch_size) % 100000 == 0:
        print(f"  Progress: {min(i + batch_size, len(df)):,} / {len(df):,} ({min(i + batch_size, len(df))/len(df)*100:.1f}%)")

# Convert to dataframe and merge
features_df = pd.DataFrame(features_list)

for col in features_df.columns:
    df[col] = features_df[col].values

print(f"\nFeature extraction complete!")
print(f"   Added {len(features_df.columns)} new features")

In [ ]:
# Feature statistics
print("\n" + "=" * 70)
print("FEATURE STATISTICS")
print("=" * 70)

feature_cols = ['sentiment_compound', 'sentiment_neg', 'urgency_score', 
                'churn_intent', 'loyalty_score']

for col in feature_cols:
    print(f"\n{col}:")
    print(f"  Min:  {df[col].min():.3f}")
    print(f"  Mean: {df[col].mean():.3f}")
    print(f"  Max:  {df[col].max():.3f}")
    print(f"  Std:  {df[col].std():.3f}")

In [ ]:
print("=" * 70)
print("CHURN RISK STRATIFICATION")
print("=" * 70)

# Define risk tiers
def assign_risk_tier(churn_score):
    if churn_score >= 0.8:
        return 'Critical'
    elif churn_score >= 0.6:
        return 'High'
    elif churn_score >= 0.4:
        return 'Moderate'
    elif churn_score >= 0.3:
        return 'Low'
    else:
        return 'Minimal'

df['churn_risk_tier'] = df['churn_intent'].apply(assign_risk_tier)

# Display distribution
risk_dist = df['churn_risk_tier'].value_counts()

print("\nChurn Risk Distribution:")
print("─" * 70)

for tier in ['Critical', 'High', 'Moderate', 'Low', 'Minimal']:
    count = risk_dist.get(tier, 0)
    pct = count / len(df) * 100
    print(f"  {tier:10s}: {count:8,} ({pct:5.1f}%)")

print("─" * 70)
print(f"  Total:       {len(df):8,} (100.0%)")

In [ ]:
print("=" * 70)
print("CREATING VISUALIZATIONS")
print("=" * 70)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Plot 1: Sentiment distribution
axes[0, 0].hist(df['sentiment_compound'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df['sentiment_compound'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df['sentiment_compound'].mean():.3f}")
axes[0, 0].set_xlabel('Sentiment Compound')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Sentiment Distribution', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Plot 2: Urgency distribution
urgency_counts = df['urgency_score'].value_counts().sort_index()
axes[0, 1].bar(urgency_counts.index, urgency_counts.values, color='orange', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Urgency Score')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Urgency Distribution', fontweight='bold')
axes[0, 1].grid(alpha=0.3)

# Plot 3: Churn intent distribution
churn_counts = df['churn_intent'].value_counts().sort_index()
axes[0, 2].bar(churn_counts.index, churn_counts.values, color='red', edgecolor='black', alpha=0.7)
axes[0, 2].set_xlabel('Churn Intent')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].set_title('Churn Intent Distribution', fontweight='bold')
axes[0, 2].grid(alpha=0.3)

# Plot 4: Risk tier distribution
tier_order = ['Minimal', 'Low', 'Moderate', 'High', 'Critical']
tier_counts = df['churn_risk_tier'].value_counts().reindex(tier_order, fill_value=0)
colors = ['green', 'yellowgreen', 'orange', 'orangered', 'darkred']
axes[1, 0].barh(tier_order, tier_counts.values, color=colors, edgecolor='black', alpha=0.8)
axes[1, 0].set_xlabel('Count')
axes[1, 0].set_title('Churn Risk Tiers', fontweight='bold')
axes[1, 0].grid(alpha=0.3, axis='x')

# Plot 5: Topic distribution
topic_counts = df['dominant_topic'].value_counts().sort_index()
axes[1, 1].bar([f"T{i+1}" for i in topic_counts.index], topic_counts.values, 
               color='purple', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Topic')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Topic Distribution', fontweight='bold')
axes[1, 1].grid(alpha=0.3)

# Plot 6: Sentiment vs Churn
axes[1, 2].scatter(df['sentiment_compound'].sample(10000), 
                   df['churn_intent'].sample(10000),
                   alpha=0.3, s=1, color='navy')
axes[1, 2].set_xlabel('Sentiment Compound')
axes[1, 2].set_ylabel('Churn Intent')
axes[1, 2].set_title('Sentiment vs Churn Intent', fontweight='bold')
axes[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/nlp_features_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nSaved: outputs/nlp_features_summary.png")

### 8.1 VADER vs FinBERT

In [3]:
from transformers import pipeline
import torch

print("=" * 70)
print("VADER VALIDATION: FinBERT Comparison")
print("=" * 70)

# Sample
sample_size = 5000
sample_df = df[df['narrative_clean'].notna()].sample(n=sample_size, random_state=42).copy()

# Load FinBERT
device = 0 if torch.cuda.is_available() else -1
finbert = pipeline("sentiment-analysis", 
                   model="ProsusAI/finbert", 
                   device=device)

# Process
finbert_scores = []
for text in tqdm(sample_df['narrative_clean'], desc="FinBERT processing"):
    try:
        result = finbert(text[:512])[0]
        score = result['score'] if result['label'] == 'positive' else -result['score']
        finbert_scores.append(score)
    except:
        finbert_scores.append(0.0)

sample_df['finbert_sentiment'] = finbert_scores

# Correlation
corr = sample_df['sentiment_compound'].corr(sample_df['finbert_sentiment'])
print(f"\nPearson correlation: {corr:.3f}")
print(f"Interpretation: {'Weak' if abs(corr) < 0.3 else 'Moderate' if abs(corr) < 0.7 else 'Strong'} correlation")

C:\Users\shubh\hw_FAI\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

## 6. Risk Stratification

Create churn risk tiers based on churn_intent score.

In [ ]:
print("=" * 70)
print("CHURN RISK STRATIFICATION")
print("=" * 70)

# Define risk tiers
def assign_risk_tier(churn_score):
    if churn_score >= 0.8:
        return 'Critical'
    elif churn_score >= 0.6:
        return 'High'
    elif churn_score >= 0.4:
        return 'Moderate'
    elif churn_score >= 0.3:
        return 'Low'
    else:
        return 'Minimal'

df['churn_risk_tier'] = df['churn_intent'].apply(assign_risk_tier)

# Display distribution
risk_dist = df['churn_risk_tier'].value_counts()

print("\nChurn Risk Distribution:")
print("─" * 70)

for tier in ['Critical', 'High', 'Moderate', 'Low', 'Minimal']:
    count = risk_dist.get(tier, 0)
    pct = count / len(df) * 100
    print(f"  {tier:10s}: {count:8,} ({pct:5.1f}%)")

print("─" * 70)
print(f"  Total:       {len(df):8,} (100.0%)")

---
## 7. Save final dataset for next stage

In [ ]:
print("=" * 70)
print("SAVING FINAL DATASET")
print("=" * 70)

# Final columns to keep
final_columns = [
    'Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
    'Company', 'State', 'ZIP code', 'Submitted via', 'Company response to consumer',
    'Timely response?', 'Consumer disputed?',
    'narrative_clean',
    'sentiment_compound', 'sentiment_neg', 'sentiment_neu', 'sentiment_pos',
    'urgency_score', 'churn_intent', 'loyalty_score',
    'text_length', 'word_count',
    'dominant_topic', 'churn_risk_tier'
]

# Keep only existing columns
final_columns = [col for col in final_columns if col in df.columns]

df_final = df[final_columns].copy()

print(f"\nFinal dataset shape: {df_final.shape}")
print(f"  Rows: {df_final.shape[0]:,}")
print(f"  Columns: {df_final.shape[1]}")

# Save
output_path = 'data/cfpb_complaints_with_nlp_features.csv'
df_final.to_csv(output_path, index=False)

print(f"\nSaved: {output_path}")
print(f"   File size: {df_final.memory_usage(deep=True).sum() / 1024**2:.1f} MB (in memory)")

In [ ]:
# Save summary statistics
summary = {
    'dataset_info': {
        'total_complaints': len(df_final),
        'date_range': f"{df_final['Date received'].min()} to {df_final['Date received'].max()}",
        'features': len(final_columns)
    },
    'nlp_features': {
        'sentiment_mean': float(df_final['sentiment_compound'].mean()),
        'urgency_mean': float(df_final['urgency_score'].mean()),
        'churn_intent_mean': float(df_final['churn_intent'].mean()),
        'loyalty_mean': float(df_final['loyalty_score'].mean())
    },
    'risk_tiers': {
        tier: int(count) 
        for tier, count in df_final['churn_risk_tier'].value_counts().items()
    },
    'topic_distribution': {
        f'Topic_{i+1}': int(count)
        for i, count in df_final['dominant_topic'].value_counts().sort_index().items()
    },
    'timestamp': datetime.now().isoformat()
}

with open('outputs/nlp_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\nSaved: outputs/nlp_summary.json")

In [ ]:
print("\n" + "=" * 70)
print("NLP FEATURE ENGINEERING COMPLETE")
print("=" * 70)

print("\nSummary:")
print(f"  Total complaints processed:  {len(df_final):,}")
print(f"  NLP features extracted:      {len([c for c in final_columns if 'sentiment' in c or 'urgency' in c or 'churn' in c or 'loyalty' in c])}")
print(f"  Topics discovered:           5")
print(f"  Topic model used:            {winner.upper()}")
print(f"  Risk tiers created:          5")

print("\nOutput files:")
print(f"  data/cfpb_complaints_with_nlp_features.csv")
print(f"  models/tfidf_vectorizer.pkl")
print(f"  models/lda_final_model.pkl")
print(f"  models/lda_final_vectorizer.pkl")
print(f"  outputs/nlp_features_summary.png")
print(f"  outputs/nlp_summary.json")